# CharWriteViT on Kaggle
Stage 1 requires only `crawford/emnist`. Licensed handwriting fonts are optional and add punctuation supervision. IAM is attached later for OCR and generator training.

In [ ]:
REPOSITORY = 'https://github.com/YOUR_USER/YOUR_REPO.git'
!git clone $REPOSITORY /kaggle/working/CharWriteViT
%cd /kaggle/working/CharWriteViT
!pip install -q -e .

In [ ]:
# Attach crawford/emnist. Licensed fonts are optional in this stage.
!find /kaggle/input -maxdepth 4 -type f | head -80
!python -m charwritevit.cli.prepare_char_data --config configs/kaggle.yaml

In [ ]:
# Smoke test before spending a full Kaggle session.
from pathlib import Path
import subprocess, sys
from IPython.display import display, Image

PROJECT = Path('/kaggle/working/CharWriteViT')
SMOKE_OUTPUT = PROJECT / 'outputs' / 'classifier-smoke'
subprocess.run([sys.executable, '-m', 'charwritevit.cli.train_classifier',
                '--config', str(PROJECT / 'configs/kaggle.yaml'),
                '--output', str(SMOKE_OUTPUT), '--epochs', '1',
                '--max-steps', '20'], cwd=PROJECT, check=True)

for artifact in [SMOKE_OUTPUT / 'classifier_curves.png',
                 SMOKE_OUTPUT / 'preview' / 'epoch_000.png']:
    if not artifact.is_file():
        raise RuntimeError(f'Training exited successfully but did not create {artifact}')
    display(Image(filename=str(artifact)))

In [ ]:
# Full classifier training. Save best.pt as a Kaggle output before continuing.
CLASSIFIER_OUTPUT = PROJECT / 'outputs' / 'classifier'
subprocess.run([sys.executable, '-m', 'charwritevit.cli.train_classifier',
                '--config', str(PROJECT / 'configs/kaggle.yaml'),
                '--output', str(CLASSIFIER_OUTPUT)], cwd=PROJECT, check=True)
curve = CLASSIFIER_OUTPUT / 'classifier_curves.png'
if not curve.is_file():
    raise RuntimeError(f'Training exited successfully but did not create {curve}')
display(Image(filename=str(curve)))

In [ ]:
# STOP: this notebook currently covers stage 1 only.
print('Stage 1 ends after classifier training and inspection.')
print('Do not run IAM/OCR/generator stages until IAM and licensed fonts are attached and their inputs are validated.')

In [ ]:
# Archive checkpoints so the notebook Output can be saved as a Kaggle Dataset.
!tar -czf /kaggle/working/charwritevit-checkpoints.tgz outputs configs/kaggle.yaml